In [ ]:
from google.colab import drive
from zipfile import ZipFile

drive.mount("/content/drive")

with ZipFile(
    "/content/drive/MyDrive/llm-post-training/sellerops-sft-cloud.zip"
) as archive:
    archive.extractall("/content")

Mounted at /content/drive


# SellerOps: full supervised fine-tuning (guided practice)

Train **all parameters** of Qwen2.5-0.5B-Instruct on our 440 training decisions.
SFT learns the next assistant action from a conversation; it does not train the tool itself.
Input: system instructions + previous messages. Target: the last assistant JSON action,
including its end-of-message token. Output: a complete model checkpoint and a development comparison.

**Four exercises:** prompt/target separation, loss masking, label padding, and accuracy.
Replace each `...` marked `FILL_ME`. Hints explain Python syntax; assertions check your work.
An intentional `NotImplementedError` means finish that exercise before continuing.
Run cells in order. Every code cell has a guide immediately above it.

**Cloud preparation:** upload the `sellerops-agent` folder (at least `src`, `data/sft`,
`data/manifest.json`, and this notebook). Frozen task files are not needed.
Enable a GPU and internet for the pinned model download. Set `PROJECT_ROOT` below.
Use a writable output directory: `/content/sellerops-runs` in Colab or
`/kaggle/working/sellerops-runs` in Kaggle. Download results before the session ends.

**Environment:** Colab may use its managed Python runtime (explicitly authorized on
September 6). Run the hardware audit, then the Colab dependency setup cell below.
It preserves the installed PyTorch build and pins the three training libraries.
Restart the Python session after installation, then rerun the audit and continue to
section 2. The setup cell becomes a no-op when its versions already match.
Local work still uses the `llm-post-training` Conda environment. An optional CUDA
Conda configuration remains in `configs/environment.cuda.yml` for other hosts.
Cloud requirements are recorded in `configs/requirements.colab.txt`; the exact
installed environment is exported with each run.

The notebook open in Colab is your working copy. The extracted notebook in the ZIP
is a separate file and does not synchronize with edits in the Colab editor. Save
your working notebook to Drive. Keep source/data in `/content/sellerops-agent`;
`OUTPUT_ROOT` may point to Drive to preserve completed checkpoints.

This notebook does not publish models or load the frozen test set.


## 1. Audit hardware before choosing packages

**Purpose:** identify the actual accelerator, Python executable, and Conda environment.
`shutil.which` finds a command; `subprocess.run` executes it without a shell.
This cell uses only the standard library and can run in the initial hosted kernel.
A GPU listing alone does not prove that PyTorch can use CUDA; we check that next.


In [ ]:
import os, sys, platform, shutil, subprocess
print("OS:", platform.platform())
print("Python:", sys.executable)
print("Conda:", os.environ.get("CONDA_DEFAULT_ENV"), shutil.which("conda"))
if shutil.which("conda"):
    subprocess.run(["conda", "env", "list"], check=False)
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No nvidia-smi. Enable a cloud GPU before training.")


OS: Linux-6.6.122+-x86_64-with-glibc2.39
Python: /usr/bin/python3
Conda: None None


## 1b. Colab dependency setup — before training imports

**Purpose:** install only the pinned training libraries into the authorized Colab
runtime. The previous cell must show the actual GPU/OS first. PyTorch is checked
for CUDA support here and is not replaced. A pip constraint fixes it to the exact
installed version; an incompatible dependency resolution stops instead of upgrading it.

`sys.executable -m pip` targets the current Python interpreter. `check=True` stops
on a failed command. A dictionary stores required package/version pairs; only
mismatches trigger installation. No manual `pip install torch` is needed.

After a successful install, **restart the Python session** using the Runtime menu
(or Colab's restart prompt), then rerun the audit and this cell before section 2.
A restart clears variables; files normally remain on that VM. If you replace the
VM entirely, remount Drive and extract the project ZIP again.


In [ ]:
import sys, os, importlib.util, importlib.metadata, subprocess
from pathlib import Path

def running_in_colab():
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False

IS_COLAB = running_in_colab()
if IS_COLAB:
    import torch
    print("Preserving PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
    assert torch.cuda.is_available(), "Enable a GPU runtime before setup."
    required = {"transformers": "5.14.1", "datasets": "5.0.0", "accelerate": "1.14.0"}
    def installed_version(package):
        try:
            return importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            return None
    changes = {name: version for name, version in required.items()
               if installed_version(name) != version}
    if changes:
        constraint = Path("/content/sellerops-torch-constraint.txt")
        constraint.write_text(f"torch=={importlib.metadata.version('torch')}\n")
        print("Installing pinned training libraries:", required)
        subprocess.run([sys.executable, "-m", "pip", "install", "--constraint", str(constraint),
                        *[f"{name}=={version}" for name, version in required.items()]], check=True)
        raise RuntimeError("Setup complete. Restart the Python session, then rerun from the audit cell.")
    print("Required versions already match. Continue to section 2.")
else:
    assert Path(sys.prefix).name == "llm-post-training", "Select the llm-post-training environment locally."
    print("Using existing local Conda environment; no installation performed.")


Preserving PyTorch: 2.11.0+cu128 CUDA: 12.8
Required versions already match. Continue to section 2.


## 2. Imports and runtime checks

**Purpose:** verify the authorized runtime and CUDA before allocating weights.
No dependencies are installed here. `assert` stops execution with a useful message.
`is_bf16_supported()` selects BF16 where supported; other GPUs use FP16 autocast.
Weights remain FP32 for stable optimizer updates; Trainer handles mixed precision.


In [ ]:
from pathlib import Path
assert IS_COLAB or Path(sys.prefix).name == "llm-post-training", (
    "Use Colab managed Python or the local llm-post-training environment."
)
import gc, json, hashlib, time, math, importlib.metadata
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, set_seed
assert torch.cuda.is_available(), "This training notebook requires a CUDA GPU."
USE_BF16 = torch.cuda.is_bf16_supported()
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("Mixed precision:", "bf16" if USE_BF16 else "fp16")


GPU: Tesla T4
VRAM GiB: 14.56
PyTorch: 2.11.0+cu128 CUDA runtime: 12.8
Mixed precision: bf16


## 3. Paths and experiment configuration

**Edit:** point `PROJECT_ROOT` to your uploaded folder. `Path` joins paths with `/`.
The default model/revision matches a saved project baseline. Keep it fixed for this experiment.
A timestamp creates a new output directory. The effective batch is microbatch × accumulation
on this single-GPU notebook. Start with one epoch; use development evidence before adding more.
`MAX_LENGTH` is a guard: we reject oversized examples instead of silently cutting off targets.


In [ ]:
PROJECT_ROOT = Path("/content/sellerops-agent")  # EDIT for your upload location
OUTPUT_ROOT = Path("/content/sellerops-runs")  # Kaggle: /kaggle/working/sellerops-runs
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
SEED = 20260906
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 256
LEARNING_RATE = 2e-5
EPOCHS = 2
MICROBATCH = 1
ACCUMULATION = 8
assert (PROJECT_ROOT / "src/sellerops/schemas.py").is_file(), "Fix PROJECT_ROOT."
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from sellerops.schemas import validate_action, SchemaError
RUN_DIR = OUTPUT_ROOT / time.strftime("%Y%m%d-%H%M%S-full-sft")
RUN_DIR.mkdir(parents=True, exist_ok=False)
set_seed(SEED)
print("Run:", RUN_DIR, "Effective batch:", MICROBATCH * ACCUMULATION)


Run: /content/sellerops-runs/20260917-234821-full-sft Effective batch: 8


## 4. Load and verify train/development records

**Purpose:** verify file hashes, split isolation, schema, and target position.
JSONL has one JSON object per line. A set comprehension collects unique IDs.
The `&` operator finds overlap between sets; an empty intersection prevents leakage.
We read only SFT train/dev files and the manifest, never frozen task contents.


In [ ]:
manifest = json.loads((PROJECT_ROOT / "data/manifest.json").read_text())
def load_records(split):
    relative = f"sft/{split}.jsonl"
    raw = (PROJECT_ROOT / "data" / relative).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == manifest["file_sha256"][relative]
    rows = [json.loads(line) for line in raw.decode().splitlines() if line.strip()]
    assert len(rows) == manifest["sft_record_counts"][split]
    for row in rows:
        assert row["split"] == split
        assert row["dataset_version"] == manifest["dataset_version"]
        assert row["target_message_index"] == len(row["messages"]) - 1
        assert row["messages"][-1]["role"] == "assistant"
        validate_action(json.loads(row["messages"][-1]["content"]))
    return rows
train_records, dev_records = load_records("train"), load_records("dev")
for key in ("record_id", "task_id", "template_id"):
    assert not ({r[key] for r in train_records} & {r[key] for r in dev_records})
print("Train:", len(train_records), "Dev:", len(dev_records))
print(json.dumps(train_records[0], ensure_ascii=False, indent=2))


Train: 440 Dev: 78
{
  "dataset_version": "sellerops-sft-v0.2",
  "environment_version": "sellerops-sim-v0.1",
  "family": "product_search",
  "locale": "id",
  "messages": [
    {
      "content": "You are the SellerOps decision policy.\n\nReturn exactly one valid JSON object and nothing else. Do not return Markdown,\ncode fences, explanations, reasoning, or multiple JSON values.\n\nThe available actions and their exact schemas are:\n\n1. Call one tool:\n{\"type\":\"tool_call\",\"tool\":\"<tool_name>\",\"arguments\":{...}}\n\nAvailable tools:\n- search_products: Find products by query and optional filters.\n  Required arguments: query (string) and filters (object). Filters may contain\n  only brand (string), category (string), color (string), in_stock (boolean),\n  and max_price_idr (integer).\n- get_inventory: Retrieve current inventory for one known product.\n  Required arguments: product_id (string).\n- get_discount_policy: Retrieve the discount policy for one known product.\n  Req

## 5. Load the tokenizer

**Purpose:** turn messages into the exact text convention expected by the model.
The tokenizer is downloaded at the same pinned revision as the weights.
We use right padding for training. Padding positions will have attention 0 and label -100.
`enable_thinking=False` is recorded consistently; the chosen Qwen2.5 template does not use it.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
assert tokenizer.chat_template, "This notebook requires a chat template."
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("EOS:", tokenizer.eos_token, "PAD:", tokenizer.pad_token)


EOS: <|im_end|> PAD: <|endoftext|>


## 6. Exercise 1 — separate context and target

**Goal:** return `(context_messages, target_message)` using `target_message_index`.
All messages before that index are context, including earlier assistant messages.
Only the final assistant message is the target.

**Syntax hint:** `items[:k]` selects elements before index `k`; `items[k]` selects one item.
Replace the two ellipses. Do not modify the original record.


In [ ]:
def split_target(record):
    messages = record["messages"]
    index = record["target_message_index"]
    context = messages[:index]
    target = messages[index]
    if context is Ellipsis or target is Ellipsis:
        raise NotImplementedError("Exercise 1: separate context and target.")
    assert target["role"] == "assistant"
    return context, target

example = train_records[0]
context, target = split_target(example)
assert context + [target] == example["messages"]
assert len(context) == example["target_message_index"]
assert split_target({"messages": [1, 2, {"role": "assistant"}], "target_message_index": 2})[0] == [1, 2]
print("Exercise 1 passed.")


Exercise 1 passed.


## 7. Render and tokenize without truncating

**Purpose:** find the exact token boundary where the answer starts.
The context template uses `add_generation_prompt=True` to open the assistant turn.
The full conversation uses `False` because the assistant answer is already present.
We tokenize the complete rendered strings with `add_special_tokens=False` so special tokens
are not duplicated. The prefix assertion catches incompatible template/token boundaries.
The suffix includes the target JSON and end-of-message marker; never shift labels manually.


In [ ]:
def render_tokens(record):
    context, target = split_target(record)
    prompt = tokenizer.apply_chat_template(
        context, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    complete = tokenizer.apply_chat_template(
        context + [target], tokenize=False, add_generation_prompt=False, enable_thinking=False)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(complete, add_special_tokens=False)["input_ids"]
    assert full_ids[:len(prompt_ids)] == prompt_ids, "Template prefix mismatch; inspect rendered text."
    assert 0 < len(prompt_ids) < len(full_ids)
    assert len(full_ids) <= MAX_LENGTH, (
        f"{record['record_id']}: {len(full_ids)} tokens exceeds MAX_LENGTH. "
        "Inspect and increase the limit if memory permits; do not truncate the target."
    )
    return full_ids, len(prompt_ids)

ids, boundary = render_tokens(example)
print("CONTEXT:\n", tokenizer.decode(ids[:boundary]))
print("TARGET:\n", tokenizer.decode(ids[boundary:]))


CONTEXT:
 <|im_start|>system
You are the SellerOps decision policy.

Return exactly one valid JSON object and nothing else. Do not return Markdown,
code fences, explanations, reasoning, or multiple JSON values.

The available actions and their exact schemas are:

1. Call one tool:
{"type":"tool_call","tool":"<tool_name>","arguments":{...}}

Available tools:
- search_products: Find products by query and optional filters.
  Required arguments: query (string) and filters (object). Filters may contain
  only brand (string), category (string), color (string), in_stock (boolean),
  and max_price_idr (integer).
- get_inventory: Retrieve current inventory for one known product.
  Required arguments: product_id (string).
- get_discount_policy: Retrieve the discount policy for one known product.
  Required arguments: product_id (string).
- calculate_offer: Calculate a policy-compliant offer for one known product.
  Required arguments: product_id (string), quantity (integer of at least 1),
  and 

## 8. Exercise 2 — assistant-only loss masking

**Goal:** create labels of equal length to `input_ids`: context positions are `-100`,
target positions retain their token IDs. `-100` is the cross-entropy ignore index.
Earlier assistant turns remain context, so they must be masked too.

**Syntax hint:** `[value] * n` repeats a value; `a + b` joins lists;
`items[k:]` selects the suffix. Return a new list, not a modified input.
Causal-LM models shift predictions/labels internally to predict the next token.


In [ ]:
def make_labels(input_ids, prompt_length):
    labels = [-100] * prompt_length + input_ids[prompt_length:]
    if labels is Ellipsis:
        raise NotImplementedError("Exercise 2: mask prompt labels.")
    return labels

original = [11, 12, 13, 14, 15]
assert make_labels(original, 3) == [-100, -100, -100, 14, 15]
assert original == [11, 12, 13, 14, 15]
assert make_labels([8, 9], 1) == [-100, 9]
print("Exercise 2 passed.")


Exercise 2 passed.


## 9. Prepare datasets and visually inspect supervision

**Purpose:** apply your functions to every record and verify every boundary.
A list comprehension maps a function over records. `Dataset.from_list` creates the format
Trainer consumes. The printed decoded supervised text must contain only the final action
and its closing marker, never the system instructions or tool observations.


In [ ]:
def encode_record(record):
    ids, boundary = render_tokens(record)
    labels = make_labels(ids, boundary)
    assert len(labels) == len(ids)
    assert labels[:boundary] == [-100] * boundary
    assert labels[boundary:] == ids[boundary:]
    assert any(label != -100 for label in labels)
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_dataset = Dataset.from_list([encode_record(r) for r in train_records])
dev_dataset = Dataset.from_list([encode_record(r) for r in dev_records])
lengths = sorted(len(r["input_ids"]) for r in train_dataset)
print("Train token lengths min/median/max:", lengths[0], lengths[len(lengths)//2], lengths[-1])
for index in (0, len(train_dataset)//2, len(train_dataset)-1):
    row = train_dataset[index]
    print("Supervised target:", tokenizer.decode([x for x in row["labels"] if x != -100]))


Train token lengths min/median/max: 470 562 825
Supervised target: {"arguments":{"filters":{"category":"accessories","in_stock":true,"max_price_idr":249000},"query":"accessories"},"tool":"search_products","type":"tool_call"}<|im_end|>

Supervised target: {"arguments":{"product_id":"P1008"},"tool":"get_inventory","type":"tool_call"}<|im_end|>

Supervised target: {"data":{"discount_eligible":true,"max_discount_percent":5,"product_id":"P1011","source":"discount-policy-v1#P1011"},"message":"Produk P1011 dapat diberi diskon; batasnya 5% menurut discount-policy-v1#P1011.","status":"success","type":"final"}<|im_end|>



## 10. Exercise 3 — pad labels safely

**Goal:** right-pad each label list to `width` using the ignore index.
Different examples have different lengths, but tensors in a batch need a common shape.
Token padding uses the PAD token; **label padding uses -100**, not the PAD token ID.

**Syntax hint:** the number of padding entries is `width - len(labels)`.
`torch.tensor(..., dtype=torch.long)` creates integer tensors for token IDs and labels.
We pad by position, so real EOS tokens remain supervised even if PAD and EOS share an ID.


In [ ]:
def pad_labels(labels, width):
    assert width >= len(labels)
    padded = labels + [-100] * (width - len(labels))
    if padded is Ellipsis:
        raise NotImplementedError("Exercise 3: pad labels.")
    return padded

assert pad_labels([-100, 7], 4) == [-100, 7, -100, -100]
assert pad_labels([9], 1) == [9]

def collate(features):
    width = max(len(f["input_ids"]) for f in features)
    return {
        "input_ids": torch.tensor([f["input_ids"] + [tokenizer.pad_token_id] * (width-len(f["input_ids"])) for f in features], dtype=torch.long),
        "attention_mask": torch.tensor([f["attention_mask"] + [0] * (width-len(f["input_ids"])) for f in features], dtype=torch.long),
        "labels": torch.tensor([pad_labels(f["labels"], width) for f in features], dtype=torch.long),
    }

batch = collate([train_dataset[0], train_dataset[-1]])
assert batch["labels"].shape == batch["input_ids"].shape
assert torch.all(batch["labels"][batch["attention_mask"] == 0] == -100)
for i, row in enumerate([train_dataset[0], train_dataset[-1]]):
    assert batch["labels"][i, :len(row["labels"])].tolist() == row["labels"]
print("Exercise 3 passed. Batch shape:", batch["input_ids"].shape)


Exercise 3 passed. Batch shape: torch.Size([2, 626])


## 11. Exercise 4 — calculate applicable accuracy

**Goal:** return the fraction of `True` values after excluding `None`.
`None` means a metric does not apply (for example, tool choice for a final answer).
An empty eligible list returns `None`, not a misleading zero.

**Syntax hint:** `sum` counts Python booleans because `True` behaves like 1;
`len` counts entries. The denominator must use eligible values only.


In [ ]:
def accuracy(values):
    eligible = [v for v in values if v is not None]
    if not eligible:
        return None
    rate = sum(eligible)/len(eligible)
    if rate is Ellipsis:
        raise NotImplementedError("Exercise 4: calculate accuracy.")
    return rate

assert accuracy([True, False, None, True]) == 2/3
assert accuracy([None]) is None
assert accuracy([False]) == 0.0
print("Exercise 4 passed.")


Exercise 4 passed.


## 12. Strict development evaluation

**Purpose:** compare generated actions against the same 78 development decisions.
`json.loads` rejects prefixes/fences/multiple JSON objects. The project validator rejects
invalid action schemas. Invalid predictions count as failures; no repair hides errors.
Tool/argument metrics apply only to expected tool calls. Exact action includes final wording,
so it is stricter than business-task success. We save raw outputs for failure inspection.
`model.eval()` disables dropout; `torch.inference_mode()` disables gradient tracking.
Greedy batch-1 decoding and the same tokenizer are used for both base and SFT.


In [ ]:
def score_prediction(record, text):
    expected = json.loads(record["messages"][-1]["content"])
    action, json_valid, schema_valid, error = None, False, False, None
    try:
        action = json.loads(text)
        json_valid = True
        validate_action(action)
        schema_valid = True
    except (ValueError, TypeError, SchemaError) as exc:
        error = str(exc)
    obj = action if isinstance(action, dict) else {}
    is_tool = expected["type"] == "tool_call"
    return {
        "record_id": record["record_id"], "raw_prediction": text,
        "expected_action": expected, "error": error,
        "json_valid": json_valid, "schema_valid": schema_valid,
        "action_type_correct": obj.get("type") == expected["type"],
        "tool_choice_correct": (obj.get("type") == "tool_call" and obj.get("tool") == expected["tool"]) if is_tool else None,
        "arguments_exact": (obj.get("type") == "tool_call" and obj.get("tool") == expected["tool"] and obj.get("arguments") == expected["arguments"]) if is_tool else None,
        "exact_action_match": schema_valid and action == expected,
    }

@torch.inference_mode()
def evaluate_actions(model, label):
    model.eval()
    rows = []
    started = time.perf_counter()
    for index, record in enumerate(dev_records):
        context, _ = split_target(record)
        text = tokenizer.apply_chat_template(context, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt").to(model.device)
        with torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16):
            generated = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS,
                                       pad_token_id=tokenizer.pad_token_id, use_cache=True)
        prediction = tokenizer.decode(generated[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        rows.append(score_prediction(record, prediction))
        if (index + 1) % 10 == 0:
            print(label, index + 1, "/", len(dev_records))
    keys = ("json_valid", "schema_valid", "action_type_correct", "tool_choice_correct", "arguments_exact", "exact_action_match")
    summary = {k: {"rate": accuracy([r[k] for r in rows]), "correct": sum(r[k] is True for r in rows),
                   "total": sum(r[k] is not None for r in rows)} for k in keys}
    result = {"label": label, "record_count": len(rows), "metrics": summary, "seconds": time.perf_counter()-started}
    (RUN_DIR / f"{label}.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False)+"\n" for r in rows))
    (RUN_DIR / f"{label}.summary.json").write_text(json.dumps(result, indent=2))
    return result, rows

perfect = score_prediction(dev_records[0], dev_records[0]["messages"][-1]["content"])
assert perfect["schema_valid"] and perfect["exact_action_match"]
assert not score_prediction(dev_records[0], "answer: {} ")["json_valid"]
assert not score_prediction(dev_records[0], '{"type":"unknown"}')["schema_valid"]
print("Evaluator checks passed.")


Evaluator checks passed.


## 13. Load all trainable weights and save provenance

**Purpose:** load a complete model without LoRA or quantization.
`requires_grad` decides whether a parameter receives gradients. We assert every parameter
is trainable. FP32 weights plus autocast support stable AdamW mixed-precision training.
We save configuration, data hashes, template, and installed package versions before training.
Seed control improves repeatability; identical results across different GPUs are not guaranteed.


In [ ]:
def load_fresh_model():
    set_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, dtype=torch.float32, attn_implementation="sdpa")
    model.config.use_cache = False
    model.to("cuda:0")
    assert all(p.requires_grad for p in model.parameters())
    print("Trainable parameters:", sum(p.numel() for p in model.parameters()))
    return model

versions = {p: importlib.metadata.version(p) for p in ("torch", "transformers", "datasets", "accelerate")}
config = dict(model_id=MODEL_ID, model_revision=MODEL_REVISION, seed=SEED,
              max_length=MAX_LENGTH, max_new_tokens=MAX_NEW_TOKENS, learning_rate=LEARNING_RATE,
              epochs=EPOCHS, microbatch=MICROBATCH, accumulation=ACCUMULATION,
              full_finetuning=True, bf16=USE_BF16, fp16=not USE_BF16,
              dataset_version=manifest["dataset_version"], data_hashes=manifest["file_sha256"],
              gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda, versions=versions,
              generation={"do_sample": False, "batch_size": 1, "enable_thinking": False})
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
(RUN_DIR / "chat_template.jinja").write_text(tokenizer.chat_template)
(RUN_DIR / "pip-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
if not IS_COLAB and Path(sys.prefix).name == "llm-post-training" and shutil.which("conda"):
    (RUN_DIR / "environment-resolved.yml").write_text(subprocess.check_output(["conda", "env", "export", "-n", "llm-post-training"], text=True))
model = load_fresh_model()
base_summary, base_rows = evaluate_actions(model, "base-dev")
print(json.dumps(base_summary, indent=2))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Trainable parameters: 494032768
base-dev 10 / 78
base-dev 20 / 78
base-dev 30 / 78
base-dev 40 / 78
base-dev 50 / 78
base-dev 60 / 78
base-dev 70 / 78
{
  "label": "base-dev",
  "record_count": 78,
  "metrics": {
    "json_valid": {
      "rate": 0.6153846153846154,
      "correct": 48,
      "total": 78
    },
    "schema_valid": {
      "rate": 0.19230769230769232,
      "correct": 15,
      "total": 78
    },
    "action_type_correct": {
      "rate": 0.5,
      "correct": 39,
      "total": 78
    },
    "tool_choice_correct": {
      "rate": 0.8125,
      "correct": 39,
      "total": 48
    },
    "arguments_exact": {
      "rate": 0.0,
      "correct": 0,
      "total": 48
    },
    "exact_action_match": {
      "rate": 0.0,
      "correct": 0,
      "total": 78
    }
  },
  "seconds": 128.0386452329999
}


## 14. Configure Trainer

**Purpose:** share identical training settings between smoke and full runs.
Gradient checkpointing recomputes activations to save VRAM. Accumulation combines several
microbatches before an optimizer step. AdamW updates all parameters; clipping bounds gradient norm.
We evaluate loss without keeping vocabulary-sized logits. Full runs save one checkpoint per epoch
with optimizer state for resuming. `report_to="none"` disables external experiment services.
The smoke test uses two optimizer steps and a tiny subset, with no checkpoint save.


In [ ]:
def make_trainer(model, smoke=False):
    args = TrainingArguments(
        output_dir=str(RUN_DIR / ("smoke" if smoke else "checkpoints")),
        per_device_train_batch_size=MICROBATCH, per_device_eval_batch_size=1,
        gradient_accumulation_steps=ACCUMULATION,
        learning_rate=LEARNING_RATE, num_train_epochs=EPOCHS,
        max_steps=2 if smoke else -1, optim="adamw_torch",
        weight_decay=0.01, warmup_steps=0, max_grad_norm=1.0,
        bf16=USE_BF16, fp16=not USE_BF16,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="no" if smoke else "epoch",
        save_strategy="no" if smoke else "epoch", save_total_limit=1,
        logging_steps=1 if smoke else 5, logging_first_step=True,
        logging_nan_inf_filter=False, prediction_loss_only=True,
        report_to="none", seed=SEED, data_seed=SEED,
        dataloader_num_workers=0,
    )
    return Trainer(model=model, args=args, processing_class=tokenizer,
                   data_collator=collate,
                   train_dataset=train_dataset.select(range(min(16, len(train_dataset)))) if smoke else train_dataset,
                   eval_dataset=dev_dataset)


## 15. Two-step smoke test and memory check

**Purpose:** exercise forward, backward, and AdamW state allocation before a longer run.
Peak VRAM after an optimizer step is more useful than model size alone.
We also check that a trainable parameter changed. This is a pipeline check, not evidence of quality.
If out of memory: restart the kernel to release references; keep batch size 1, inspect lengths,
and reduce the sequence limit only if every complete example still fits. Accumulation does not
reduce the memory used by a single long example or by optimizer states.
After this cell, the smoke-updated weights are discarded before the actual experiment.


In [ ]:
probe_name, probe = next((n, p) for n, p in model.named_parameters() if n.endswith("q_proj.weight"))
before = probe.detach().float().cpu().clone()
torch.cuda.reset_peak_memory_stats()
smoke_trainer = make_trainer(model, smoke=True)
smoke_result = smoke_trainer.train()
assert math.isfinite(smoke_result.training_loss)
assert not torch.equal(before, probe.detach().float().cpu()), "Weights did not update. Inspect gradients/scaler."
smoke_memory = {"peak_allocated_gib": torch.cuda.max_memory_allocated()/2**30,
                "peak_reserved_gib": torch.cuda.max_memory_reserved()/2**30,
                "loss": smoke_result.training_loss, "probe": probe_name}
(RUN_DIR / "smoke.json").write_text(json.dumps(smoke_memory, indent=2))
print(smoke_memory)
del before, probe, smoke_trainer, model
gc.collect()
torch.cuda.empty_cache()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.663290
2,0.677040


{'peak_allocated_gib': 11.087152481079102, 'peak_reserved_gib': 13.064453125, 'loss': 0.6701652705669403, 'probe': 'model.layers.0.self_attn.q_proj.weight'}


## 16. Full SFT from fresh pretrained weights

**Purpose:** train the real experiment without contamination from the smoke updates.
A fresh model resets weights; the seed resets data ordering. `train()` runs the optimization loop.
For an interrupted full run, set `RESUME_CHECKPOINT` to its checkpoint directory and restore
this run's original config/output path; do not silently resume using different data or settings.
An epoch means one pass through the training dataset. Dev data contributes no training gradients.


In [ ]:
RESUME_CHECKPOINT = None # Optional path to a checkpoint from this exact experiment
model = load_fresh_model()
trainer = make_trainer(model)
train_result = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
assert math.isfinite(train_result.training_loss)
dev_loss = trainer.evaluate()
assert math.isfinite(dev_loss["eval_loss"])
(RUN_DIR / "training-metrics.json").write_text(json.dumps({**train_result.metrics, **dev_loss}, indent=2))
(RUN_DIR / "training-log.json").write_text(json.dumps(trainer.state.log_history, indent=2))
print(dev_loss)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 494032768


Epoch,Training Loss,Validation Loss
1,0.012416,0.050969
2,0.004349,0.022440


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
0.004349,0.022440,2


{'eval_loss': 0.0224396251142025}


## 17. Save a complete checkpoint and inspect loss

**Purpose:** save the full model and tokenizer, not a LoRA adapter.
The final folder can be loaded directly with `from_pretrained`.
The CSV log can be plotted in a spreadsheet without adding a plotting dependency.
Decreasing training loss with increasing development loss suggests overfitting; loss alone does
not establish improved tool use. Preserve both logs and generated decision metrics.


In [ ]:
import csv
FINAL_DIR = RUN_DIR / "final-model"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
trainer.state.save_to_json(str(RUN_DIR / "trainer-state.json"))
with (RUN_DIR / "loss-curve.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["step", "epoch", "loss", "eval_loss", "learning_rate", "grad_norm"], extrasaction="ignore")
    writer.writeheader()
    writer.writerows(trainer.state.log_history)
for row in trainer.state.log_history:
    print(row)
# Release optimizer state before loading the saved model to verify round-trip inference.
del trainer, model
gc.collect()
torch.cuda.empty_cache()
print("Saved:", FINAL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': 0.8051814436912537, 'grad_norm': 56.12411880493164, 'learning_rate': 2e-05, 'epoch': 0.01818181818181818, 'step': 1}
{'loss': 0.5310932397842407, 'grad_norm': 18.70482063293457, 'learning_rate': 1.9272727272727275e-05, 'epoch': 0.09090909090909091, 'step': 5}
{'loss': 0.14952539205551146, 'grad_norm': 17.234933853149414, 'learning_rate': 1.8363636363636367e-05, 'epoch': 0.18181818181818182, 'step': 10}
{'loss': 0.1868201971054077, 'grad_norm': 4.5059919357299805, 'learning_rate': 1.7454545454545456e-05, 'epoch': 0.2727272727272727, 'step': 15}
{'loss': 0.062445533275604245, 'grad_norm': 4.109054088592529, 'learning_rate': 1.6545454545454548e-05, 'epoch': 0.36363636363636365, 'step': 20}
{'loss': 0.05243020057678223, 'grad_norm': 2.860001564025879, 'learning_rate': 1.563636363636364e-05, 'epoch': 0.45454545454545453, 'step': 25}
{'loss': 0.04951658248901367, 'grad_norm': 1.6722846031188965, 'learning_rate': 1.4727272727272728e-05, 'epoch': 0.5454545454545454, 'step': 30}
{'loss

## 18. Reload and compare base versus SFT

**Purpose:** verify that the saved checkpoint loads and evaluate identical development inputs.
Reloading proves we are measuring the deliverable on disk. Fractions use the same denominator
for base and SFT. Compare schema, tool choice, and arguments separately to locate improvement.
These are development decision-level metrics, not frozen end-to-end task success.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(FINAL_DIR, dtype=torch.float32, attn_implementation="sdpa").to("cuda:0")
sft_summary, sft_rows = evaluate_actions(model, "sft-dev")
print(f"{'Metric':25} {'Base':>12} {'SFT':>12}")
for key in base_summary["metrics"]:
    base, sft = base_summary["metrics"][key], sft_summary["metrics"][key]
    print(f"{key:25} {str(base['correct'])+'/'+str(base['total']):>12} {str(sft['correct'])+'/'+str(sft['total']):>12}")
(RUN_DIR / "comparison.json").write_text(json.dumps({"base": base_summary, "sft": sft_summary}, indent=2))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

sft-dev 10 / 78
sft-dev 20 / 78
sft-dev 30 / 78
sft-dev 40 / 78
sft-dev 50 / 78
sft-dev 60 / 78
sft-dev 70 / 78
Metric                            Base          SFT
json_valid                       48/78        78/78
schema_valid                     15/78        78/78
action_type_correct              39/78        77/78
tool_choice_correct              39/48        48/48
arguments_exact                   0/48        38/48
exact_action_match                0/78        55/78


1528

## 19. Review failures and preserve your work

**Purpose:** inspect up to five remaining failures before changing training settings.
Distinguish formatting failures, incorrect tools/arguments, and harmless wording differences.
Write your observations in the following Markdown cell. Download the run directory from the
cloud file browser, including `final-model`, config, comparisons, and logs. Training checkpoints
also contain optimizer state and are larger; keep them if you need to resume.
Do not interpret the earlier 0%-exact baseline as proof that all outputs were useless.


In [ ]:
failures = [row for row in sft_rows if not row["exact_action_match"]]
for row in failures[:5]:
    print("Record:", row["record_id"])
    print("Prediction:", row["raw_prediction"])
    print("Expected:", json.dumps(row["expected_action"], ensure_ascii=False))
    print("Parser/schema error:", row["error"], "\n")
print("Failures:", len(failures), "/", len(sft_rows))
print("Download this folder:", RUN_DIR)


Record: dev-search-dev-brand-id-000-a00
Prediction: {"arguments":{"filters":{"category":"comet","in_stock":true},"query":"backpack"},"tool":"search_products","type":"tool_call"}
Expected: {"arguments": {"filters": {"brand": "Comet", "in_stock": true}, "query": "backpack"}, "tool": "search_products", "type": "tool_call"}
Parser/schema error: None 

Record: dev-search-dev-brand-id-001-a00
Prediction: {"arguments":{"filters":{"category":"mouse","in_stock":true},"query":"orbit"},"tool":"search_products","type":"tool_call"}
Expected: {"arguments": {"filters": {"brand": "Orbit", "in_stock": true}, "query": "mouse"}, "tool": "search_products", "type": "tool_call"}
Parser/schema error: None 

Record: dev-search-dev-brand-id-002-a00
Prediction: {"arguments":{"filters":{"category":"charger","in_stock":true},"query":"volt"},"tool":"search_products","type":"tool_call"}
Expected: {"arguments": {"filters": {"brand": "Volt", "in_stock": true}, "query": "charger"}, "tool": "search_products", "type": "

## 20. Your experiment notes

- GPU, peak memory, and training duration:
- Explain why prompt tokens receive `-100`:
- Explain why earlier assistant messages are masked in this dataset:
- Which metric improved most, and by how much?
- Describe one schema failure and one decision failure:
- Did exact action matching reject a reasonable final answer?
- What would you change in the next experiment, and why?

**Done when:** four exercise checks pass; supervised tokens are visually verified;
the smoke test updates weights with finite loss; full training finishes; the saved model
reloads; base/SFT development metrics and raw outputs are saved. No training results are
claimed until you actually run this notebook. The next project milestone is episode-level
verification and frozen evaluation after development choices are fixed.

References: [Chat templates](https://huggingface.co/docs/transformers/chat_templating),
[Trainer](https://huggingface.co/docs/transformers/main_classes/trainer),
[mixed precision](https://huggingface.co/docs/transformers/mixed_precision_training),
[gradient checkpointing](https://huggingface.co/docs/transformers/grad_checkpointing).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!cp -r /content/sellerops-runs \
    /content/drive/MyDrive/llm-post-training/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
from pathlib import Path
from shutil import make_archive

run_dir = Path(
    "/content/sellerops-runs/20260917-234821-full-sft"
)

assert run_dir.is_dir(), f"Folder tidak ditemukan: {run_dir}"

zip_path = Path(
    make_archive(
        base_name=str(run_dir),
        format="zip",
        root_dir=run_dir.parent,
        base_dir=run_dir.name,
    )
)

print("ZIP berhasil dibuat:")
print(zip_path)
print("Ukuran:", round(zip_path.stat().st_size / 1024**2, 2), "MB")

ZIP berhasil dibuat:
/content/sellerops-runs/20260917-234821-full-sft.zip
Ukuran: 6978.76 MB
